# 07 — Conversational LLM-Guided Forget Set Generation and Machine Unlearning
## 1. Purpose and research question
Can an LLM provide a natural-language interface for translating user requests into valid forget sets for machine unlearning?

Conversation → Qwen → Python validation and data operations → forget set → confirmation → Gradient Difference → evaluation.

Qwen handles language and conversation. Python remains the trusted layer for dataset calculations, row selection, validation and session control. The frozen Gradient Difference implementation remains responsible for machine unlearning.


## 2. Load the frozen experiment
### 2.1 Imports and paths


In [1]:
# Import only the libraries required by the established notebook workflow.
import copy
import json
import os
import random
import time
import warnings
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from openai import OpenAI


In [2]:
# Import only the libraries required by the established notebook workflow.
from sklearn.exceptions import InconsistentVersionWarning
from sklearn.metrics import average_precision_score, balanced_accuracy_score, f1_score, log_loss, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, Dataset

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, cwd / "final_submission", *cwd.parents] if (p / "data/final/kidney_transplant_assessments.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate final_submission")
DATA_PATH = ROOT / "data/final/kidney_transplant_assessments.csv"
SPLIT_PATH = ROOT / "processed_data/split_assignments.csv"
FEATURE_PATH = ROOT / "data/final/classifier_feature_list.json"
BASELINE_DIR = ROOT / "models/baseline"
GD_CONFIG_PATH = ROOT / "models/gradient_difference/recipient_withdrawal/configuration.json"
GD_PROVENANCE_PATH = ROOT / "results/gradient_difference/method_provenance.json"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")


In [3]:
# Qwen handles language while Python remains responsible for trusted operations.
RESULT_DIR = ROOT / "results/llm_forget_set_generation" / RUN_ID
MODEL_DIR = ROOT / "models/llm_forget_set_generation" / RUN_ID
RAW_DIR = RESULT_DIR / "raw_qwen_responses"
for directory in [RESULT_DIR, MODEL_DIR, RAW_DIR]:
    directory.mkdir(parents=True, exist_ok=False)


### 2.2 Load dataset and permanent split
The assessment table supplies candidate rows; the saved split identifies training rows; the feature contract preserves model inputs.


In [4]:
# Load frozen inputs so this stage reuses rather than regenerates earlier evidence.
assessments = pd.read_csv(DATA_PATH)
splits = pd.read_csv(SPLIT_PATH)
contract = json.loads(FEATURE_PATH.read_text())
features = contract["classifier_features"]
target = contract["target"]
assessments["original_split"] = assessments["recipient_id"].map(splits.set_index("recipient_id")["split"])
assert assessments["original_split"].notna().all()
counts = assessments["original_split"].value_counts()
display(pd.DataFrame({"Item": ["Dataset", "Train", "Validation", "Test", "Features"], "Value": [len(assessments), counts["train"], counts["validation"], counts["test"], len(features)]}))


,Item,Value
0,Dataset,60000
1,Train,42024
2,Validation,8988
3,Test,8988
4,Features,18


### 2.3 Load baseline model and preprocessor
The baseline is copied, never retrained or overwritten.


In [5]:
# Load frozen inputs so this stage reuses rather than regenerates earlier evidence.
class BaselineMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(24, 64), nn.ReLU(), nn.Dropout(0.1), nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1), nn.Linear(32, 1))
    def forward(self, inputs):
        return self.network(inputs).squeeze(1)

checkpoint = torch.load(BASELINE_DIR / "baseline_model.pt", map_location="cpu", weights_only=True)
with warnings.catch_warnings(record=True):
    warnings.simplefilter("always", InconsistentVersionWarning)
    preprocessor = joblib.load(BASELINE_DIR / "baseline_preprocessor.joblib")
def load_baseline_model():
    model = BaselineMLP()
    model.load_state_dict(copy.deepcopy(checkpoint["model_state_dict"]))
    return model.eval()


In [6]:
# Calculate the established evaluation outputs without changing model selection.
def predict(model, frame):
    matrix = preprocessor.transform(frame[features]).astype("float32")
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.from_numpy(matrix))).numpy()


### 2.4 Load frozen Gradient Difference settings


In [7]:
# Qwen handles language while Python remains responsible for trusted operations.
gd = json.loads(GD_CONFIG_PATH.read_text())
provenance = json.loads(GD_PROVENANCE_PATH.read_text())
assert provenance["original_objective"] == "-L_forget + L_retain"
assert (gd["epochs"], gd["primary_epoch"], gd["batch_size"], gd["learning_rate"], gd["gradient_clip_norm"]) == (5, 5, 512, 1e-5, 1.0)
SEED = gd["seed"]
THRESHOLD = gd["threshold"]
current_model = load_baseline_model()
forgotten_training_ids = set()
complete_scope_ids = set()
request_history = []
request_number = 1
turn_number = 1
conversation_history = []
turn_audit = []
pending_request = None


In [8]:
verification_rows = assessments[assessments["original_split"].eq("test")].sort_values("assessment_id")
baseline_predictions = predict(load_baseline_model(), verification_rows)


## 3. Define the conversational action contract
Qwen returns a natural response plus a small structured action. The structured part is retained for audit, while Python validates every field and operation before touching data.


In [9]:
# Keep this stage separate so its inputs and checks remain easy to audit.
ACTIONS = {"chat", "help", "dataset_query", "forget", "confirm", "cancel", "session_status", "reset", "exit"}
ENTITY_FIELDS = {"recipient_id", "donor_id", "hospital_id"}
ALLOWED_FIELDS = ENTITY_FIELDS | {"creatinine_mg_dl", "training_records", "training_consent_status", "retention_expiry_date"}
ALLOWED_QUANTILES = {0.25, 0.50, 0.75}
CONTRACT_KEYS = {"response", "action", "parameters"}
PARAMETER_KEYS = {"field", "operation", "value"}
FIELD_ALIASES = {"recipient": "recipient_id", "recipient_id": "recipient_id", "donor": "donor_id", "donor_id": "donor_id", "hospital": "hospital_id", "hospital_id": "hospital_id", "creatinine": "creatinine_mg_dl", "creatinine_mg_dl": "creatinine_mg_dl"}
OPERATION_ALIASES = {"=": "exact_match", "exact_match": "exact_match", ">": "greater_than", "greater_than": "greater_than", "<": "less_than", "less_than": "less_than", "quantile_above": "quantile_above", "quantile_below": "quantile_below", "quantile": "quantile", "count": "count", "exact_match_count": "count", "invalid_consent": "invalid_consent", "on_or_before": "on_or_before"}


## 4. Connect to Qwen and define language handling
Qwen interprets conversation and phrases results. It never filters the DataFrame, calculates trusted statistics or modifies model weights.


In [10]:
# Qwen handles language while Python remains responsible for trusted operations.
QWEN_BASE_URL = os.getenv("QWEN_BASE_URL") or "https://resolution-andreas-alerts-blah.trycloudflare.com/v1"
QWEN_MODEL = os.getenv("QWEN_MODEL") or "Qwen/Qwen3.6-35B-A3B"
QWEN_API_KEY = os.getenv("QWEN_API_KEY") or getpass("Enter QWEN_API_KEY (hidden): ").strip()
if not QWEN_API_KEY:
    raise RuntimeError("Qwen API key is not configured.")
client = OpenAI(base_url=QWEN_BASE_URL, api_key=QWEN_API_KEY, max_retries=0, timeout=120)


In [11]:
# Qwen handles language while Python remains responsible for trusted operations.
PROMPT = """You are the conversational interface for a kidney-transplant machine-unlearning notebook.
Return JSON with exactly three keys: response, action and parameters.
response is a concise natural reply for the user.
action is one of: chat, help, dataset_query, forget, confirm, cancel, session_status, reset or exit.
parameters is null for actions without data parameters. Otherwise it has exactly field, operation and value.
Allowed fields are recipient_id, donor_id, hospital_id, creatinine_mg_dl, training_records, training_consent_status and retention_expiry_date.
Allowed operations are exact_match, greater_than, less_than, quantile_above, quantile_below, quantile, count, invalid_consent and on_or_before.
Use numeric quantiles: lower quartile 0.25, median 0.50 and upper quartile 0.75. Do not calculate the resulting dataset value.
For example, deleting a recipient uses forget with recipient_id, exact_match and the ID. Asking for upper-quartile creatinine uses dataset_query with creatinine_mg_dl, quantile and 0.75.
Use conversation history to resolve follow-ups such as 'delete everything above that'. Prefer the semantic quantile operation when the previous turn discussed a quantile.
If a request is ambiguous or unsafe, use chat and ask a concise clarification question or explain the supported alternatives.
Never invent dataset values, select rows, execute code or claim that model weights changed. Python performs and verifies those operations.
""".strip()


In [12]:

def normalise_action(instruction):
    # Normalise harmless schema aliases before strict Python validation.
    if not isinstance(instruction, dict):
        return instruction
    normalised = instruction.copy()
    parameters = normalised.get("parameters")
    if not isinstance(parameters, dict):
        return normalised
    parameters = parameters.copy()
    field = parameters.get("field")
    operation = parameters.get("operation")
    if isinstance(field, str) and field in FIELD_ALIASES:
        parameters["field"] = FIELD_ALIASES[field]
    if isinstance(operation, str) and operation in OPERATION_ALIASES:
        parameters["operation"] = OPERATION_ALIASES[operation]
    normalised["parameters"] = parameters
    return normalised


In [13]:

def normalise_entity_value(instruction, data):
    # Resolve a short ID only when it has one unambiguous match in the frozen data.
    if not isinstance(instruction, dict) or not isinstance(instruction.get("parameters"), dict):
        return instruction
    normalised = instruction.copy()
    parameters = normalised["parameters"].copy()
    field = parameters.get("field")
    value = parameters.get("value")
    if isinstance(field, str) and field in ENTITY_FIELDS and isinstance(value, str):
        valid_ids = data[field].dropna().astype(str).unique()
        if value not in valid_ids:
            matches = [identifier for identifier in valid_ids if identifier.endswith(f"-{value}")]
            if len(matches) == 1:
                parameters["value"] = matches[0]
    normalised["parameters"] = parameters
    return normalised


In [14]:

def validate_dataset_query(parameters, data):
    # Dataset queries are limited to calculations Python can reproduce safely.
    field = parameters["field"]
    operation = parameters["operation"]
    value = parameters["value"]
    if field == "training_records" and operation == "count" and value is None:
        return []
    if field == "creatinine_mg_dl" and operation == "quantile" and isinstance(value, (int, float)) and not isinstance(value, bool) and float(value) in ALLOWED_QUANTILES:
        return []
    if field in ENTITY_FIELDS and operation == "count" and str(value) in set(data[field].astype(str)):
        return []
    return ["Unsupported or invalid dataset query."]


In [15]:

def validate_forget_parameters(parameters, data):
    # Exact Python checks prevent Qwen from choosing unsupported deletion rules.
    field = parameters["field"]
    operation = parameters["operation"]
    value = parameters["value"]
    if field in ENTITY_FIELDS:
        if operation != "exact_match":
            return ["Entity deletion requires exact_match."]
        return [] if str(value) in set(data[field].astype(str)) else ["Identifier was not found."]
    if field == "creatinine_mg_dl" and operation in {"greater_than", "less_than"}:
        return [] if isinstance(value, (int, float)) and not isinstance(value, bool) else ["Creatinine threshold must be numeric."]
    if field == "creatinine_mg_dl" and operation in {"quantile_above", "quantile_below"}:
        return [] if isinstance(value, (int, float)) and not isinstance(value, bool) and float(value) in ALLOWED_QUANTILES else ["Quantile is not allowed."]
    if field == "training_consent_status" and operation == "invalid_consent" and value is None:
        return []
    if field == "retention_expiry_date" and operation == "on_or_before" and (value is None or value == "2025-12-31"):
        return []
    return ["Unsupported deletion rule."]


In [16]:

def validate_action(instruction, data):
    # Python is the safety boundary for every structured action.
    if not isinstance(instruction, dict) or set(instruction) != CONTRACT_KEYS:
        return ["Invalid action structure."]
    if not isinstance(instruction["response"], str) or not instruction["response"].strip():
        return ["The conversational response is missing."]
    action = instruction["action"]
    parameters = instruction["parameters"]
    if not isinstance(action, str) or action not in ACTIONS:
        return ["Unsupported action."]
    if action in {"chat", "help", "confirm", "cancel", "session_status", "reset", "exit"}:
        return [] if parameters is None or parameters == {} else ["This action does not accept data parameters."]
    if not isinstance(parameters, dict) or set(parameters) != PARAMETER_KEYS:
        return ["Invalid action parameters."]
    field = parameters["field"]
    if not isinstance(field, str) or field not in ALLOWED_FIELDS:
        return ["Unsupported dataset field."]
    if action == "dataset_query":
        return validate_dataset_query(parameters, data)
    return validate_forget_parameters(parameters, data)


In [17]:

def save_qwen_response(response, stage):
    # Preserve the complete response before extracting its content.
    path = RAW_DIR / f"turn_{turn_number:03d}_{stage}.json"
    path.write_text(response.model_dump_json(indent=2))

def interpret_turn(user_message):
    # Conversation history allows follow-up requests such as 'delete everything above that'.
    pending_context = "No deletion request is pending."
    if pending_request is not None:
        pending_context = "A deletion request is pending review. A positive reply confirms it; a negative reply cancels it."
    messages = [{"role": "system", "content": PROMPT + "\n" + pending_context}]
    messages.extend(conversation_history)
    messages.append({"role": "user", "content": user_message})
    response = client.chat.completions.create(model=QWEN_MODEL, temperature=0, max_tokens=256, response_format={"type": "json_object"}, messages=messages, extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    save_qwen_response(response, "interpretation")
    return json.loads(response.choices[0].message.content)


In [18]:

def explain_trusted_result(context, trusted_result):
    # Qwen phrases values calculated by Python rather than guessing them.
    messages = [
        {"role": "system", "content": "Explain the supplied trusted Python result naturally and concisely. Use only the supplied facts and numbers. Do not claim any unverified action."},
        {"role": "user", "content": json.dumps({"context": context, "trusted_python_result": trusted_result})},
    ]
    response = client.chat.completions.create(model=QWEN_MODEL, temperature=0, max_tokens=256, messages=messages, extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    save_qwen_response(response, "explanation")
    return response.choices[0].message.content.strip()

def show_qwen(message):
    print(f"\nQwen:\n{message}")


In [19]:
# Qwen handles language while Python remains responsible for trusted operations.

def explain_and_show(context, trusted_result):
    try:
        message = explain_trusted_result(context, trusted_result)
    except Exception as error:
        message = f"The trusted Python result is shown above. A natural-language explanation was unavailable: {type(error).__name__}."
    show_qwen(message)
    return message


In [20]:
# Qwen handles language while Python remains responsible for trusted operations.

def record_turn(user_message, raw_instruction, validated_action, python_result, model_changed, forget_set_size, runtime_seconds=None):
    turn_audit.append({
        "turn_number": turn_number,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "user_message": user_message,
        "raw_qwen_interpretation": raw_instruction,
        "validated_action": validated_action,
        "python_result": python_result,
        "model_weights_changed": model_changed,
        "forget_set_size": forget_set_size,
        "runtime_seconds": runtime_seconds,
    })
    (RESULT_DIR / "conversation_audit.json").write_text(json.dumps(turn_audit, indent=2, default=str))


## 5. Example requests
The conversation accepts greetings, help and dataset questions as well as deletion, status, cancellation and reset requests. These random entity examples come only from the frozen training split and affect display only.


In [21]:
# Choose only IDs that occur in the frozen training split.
training_rows = assessments.loc[assessments["original_split"].eq("train")]
valid_recipient_ids = training_rows["recipient_id"].dropna().astype(str).unique()
valid_donor_ids = training_rows["donor_id"].dropna().astype(str).unique()
valid_hospital_ids = training_rows["hospital_id"].dropna().astype(str).unique()

# Randomness here only changes the example IDs shown to the user; it does not affect the experiment.
example_recipients = np.random.choice(valid_recipient_ids, size=min(5, len(valid_recipient_ids)), replace=False)
example_donors = np.random.choice(valid_donor_ids, size=min(5, len(valid_donor_ids)), replace=False)
example_hospitals = np.random.choice(valid_hospital_ids, size=min(5, len(valid_hospital_ids)), replace=False)

print("Example requests you can try:\n")
print("Example recipient requests:")
for number, recipient_id in enumerate(example_recipients, start=1):
    print(f'{number}. \"Delete all data for recipient {recipient_id}\"')
print("\nExample donor requests:")
for number, donor_id in enumerate(example_donors, start=1):
    print(f'{number}. \"Remove all data for donor {donor_id}\"')
print("\nExample hospital requests:")


Example requests you can try:

Example recipient requests:
1. "Delete all data for recipient V32P-R006743"
2. "Delete all data for recipient V32P-R005340"
3. "Delete all data for recipient V32P-R003030"
4. "Delete all data for recipient V32P-R007614"
5. "Delete all data for recipient V32P-R009707"

Example donor requests:
1. "Remove all data for donor V32P-DD001739"
2. "Remove all data for donor V32P-DD000099"
3. "Remove all data for donor V32P-DL001374"
4. "Remove all data for donor V32P-DL003581"
5. "Remove all data for donor V32P-DL001492"

Example hospital requests:


In [22]:
# Keep this stage separate so its inputs and checks remain easy to audit.
for number, hospital_id in enumerate(example_hospitals, start=1):
    print(f'{number}. \"Remove all data from hospital {hospital_id}\"')
print('\nFeature-based examples:')
print('- \"Forget records with creatinine above 2.0\"')
print('- \"Forget records with creatinine above the upper quartile\"')


1. "Remove all data from hospital V32P-H03"
2. "Remove all data from hospital V32P-H01"
3. "Remove all data from hospital V32P-H06"
4. "Remove all data from hospital V32P-H07"
5. "Remove all data from hospital V32P-H08"

Feature-based examples:
- "Forget records with creatinine above 2.0"
- "Forget records with creatinine above the upper quartile"


In [23]:
ENTITY_COLUMNS = {"recipient": "recipient_id", "donor": "donor_id", "hospital": "hospital_id"}
FIELD_TO_INTERNAL_TYPE = {"recipient_id": "recipient", "donor_id": "donor", "hospital_id": "hospital"}


In [24]:

def to_internal_instruction(action):
    # Keep the generic conversation contract separate from the existing row-selection rules.
    parameters = action["parameters"]
    field = parameters["field"]
    operation = parameters["operation"]
    value = parameters["value"]
    if field in FIELD_TO_INTERNAL_TYPE:
        return {"action": "forget", "type": FIELD_TO_INTERNAL_TYPE[field], "operator": None, "value": str(value)}
    if field == "creatinine_mg_dl" and operation in {"greater_than", "less_than"}:
        operator = ">" if operation == "greater_than" else "<"
        return {"action": "forget", "type": "creatinine_threshold", "operator": operator, "value": float(value)}
    if field == "creatinine_mg_dl" and operation in {"quantile_above", "quantile_below"}:
        operator = ">" if operation == "quantile_above" else "<"
        return {"action": "forget", "type": "creatinine_quantile", "operator": operator, "value": float(value)}
    if field == "training_consent_status":
        return {"action": "forget", "type": "invalid_consent", "operator": None, "value": None}
    return {"action": "forget", "type": "retention_expiry", "operator": None, "value": None}


In [25]:

def execute_dataset_query(action, data):
    # Dataset statistics are calculated by Python rather than guessed by the LLM.
    parameters = action["parameters"]
    field = parameters["field"]
    operation = parameters["operation"]
    value = parameters["value"]
    training = data.loc[data["original_split"].eq("train")]
    if field == "training_records":
        return {"query": "training record count", "training_records": int(len(training))}
    if field == "creatinine_mg_dl" and operation == "quantile":
        result = float(training["creatinine_mg_dl"].quantile(float(value)))
        return {"query": "training creatinine quantile", "quantile": float(value), "creatinine_mg_dl": result, "unit": "mg/dL"}
    matches = data.loc[data[field].astype(str).eq(str(value))]
    training_matches = matches.loc[matches["original_split"].eq("train")]
    return {"query": "entity record count", "field": field, "value": str(value), "complete_records": int(len(matches)), "training_records": int(len(training_matches))}


## 6. Trusted forget-set construction
Python converts the generic action into the existing validated selection rules. `complete_scope` contains every match; `training_forget` contains only matching training rows that influenced fitting.


In [26]:
def create_forget_set(instruction, data):
    # Python, not Qwen, selects rows.
    kind = instruction["type"]
    value = instruction["value"]
    operator = instruction["operator"]
    if kind in ENTITY_COLUMNS:
        return data[data[ENTITY_COLUMNS[kind]].astype(str).eq(value)].copy(), None
    if kind == "invalid_consent":
        return data[data["training_consent_status"].eq("Invalidated") & data["training_consent_version"].eq("RECIPIENT_V3")].copy(), None
    if kind == "retention_expiry":
        return data[pd.to_datetime(data["retention_expiry_date"]).le("2025-12-31")].copy(), None
    if kind == "creatinine_quantile":
        # Quantiles are calculated from frozen training data so evaluation data does not influence forget-set construction.
        value = float(data.loc[data["original_split"].eq("train"), "creatinine_mg_dl"].quantile(value))
    values = data["creatinine_mg_dl"]
    return data[values.gt(value) if operator == ">" else values.lt(value)].copy(), float(value)


In [27]:

def prepare_forget_request(user_message, action, data):
    # Keep the pending forget set separate until the user confirms it.
    instruction = to_internal_instruction(action)
    complete_scope, calculated_threshold = create_forget_set(instruction, data)
    training_forget = complete_scope.loc[
        complete_scope["original_split"].eq("train")
        & ~complete_scope["assessment_id"].isin(forgotten_training_ids)
    ].copy()
    if complete_scope.empty:
        return None, {"status": "stopped", "reason": "The rule matched no records."}
    if training_forget.empty:
        return None, {"status": "stopped", "reason": "The rule matched no currently eligible training records.", "complete_scope": int(len(complete_scope))}
    pending = {
        "user_message": user_message,
        "action": action,
        "instruction": instruction,
        "complete_scope": complete_scope,
        "training_forget": training_forget,
        "calculated_creatinine_threshold": calculated_threshold,
    }
    summary = {
        "status": "pending_confirmation",
        "request": user_message,
        "complete_matching_records": int(len(complete_scope)),
        "training_records_that_influenced_model": int(len(training_forget)),
        "calculated_creatinine_threshold_mg_dl": calculated_threshold,
    }
    return pending, summary


## 7. Frozen Gradient Difference implementation
Only confirmed training rows enter updates. The retained set excludes all current and previous forget rows. The objective remains negative forget BCE plus positive retain BCE for five epochs.


In [28]:
# Keep this stage separate so its inputs and checks remain easy to audit.
class PairedDataset(Dataset):
    def __init__(self, fx, fy, rx, ry, indices):
        self.fx, self.fy, self.rx, self.ry = map(torch.from_numpy, [fx, fy, rx, ry])
        self.indices = indices
    def __len__(self):
        return len(self.fx)
    def __getitem__(self, index):
        retain_index = self.indices[index]
        return self.fx[index], self.fy[index], self.rx[retain_index], self.ry[retain_index]

criterion = nn.BCEWithLogitsLoss()


In [29]:
def run_gradient_difference(starting_model, forget_frame, retain_frame):
    model = copy.deepcopy(starting_model)
    fx = preprocessor.transform(forget_frame[features]).astype("float32")
    fy = forget_frame[target].to_numpy(dtype="float32")
    rx = preprocessor.transform(retain_frame[features]).astype("float32")
    ry = retain_frame[target].to_numpy(dtype="float32")
    # Preserve Notebook 06 request-level seed behaviour.
    random.seed(SEED + request_number)
    np.random.seed(SEED + request_number)
    torch.manual_seed(SEED + request_number)
    optimizer = torch.optim.AdamW(model.parameters(), lr=gd["learning_rate"], weight_decay=gd["weight_decay"])
    started = time.perf_counter()
    for epoch in range(1, gd["epochs"] + 1):
        rng = np.random.default_rng(SEED + 10_000 * request_number + epoch)
        indices = rng.integers(0, len(retain_frame), len(forget_frame))
        loader = DataLoader(PairedDataset(fx, fy, rx, ry, indices), batch_size=gd["batch_size"], shuffle=True, generator=torch.Generator().manual_seed(SEED + 20_000 * request_number + epoch))
        model.train()
        for forget_x, forget_y, retain_x, retain_y in loader:
            optimizer.zero_grad(set_to_none=True)
            # Exact Notebook 06 objective.
            loss = -criterion(model(forget_x), forget_y) + criterion(model(retain_x), retain_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), gd["gradient_clip_norm"])
            optimizer.step()
    return model.eval(), time.perf_counter() - started

# This frozen function is called only after conversational confirmation.


## 8. Evaluation and verified session operations
Python calculates before/after behaviour, trusted session status and visible reset evidence. No unrelated full-retraining reference is used.


In [30]:
# Calculate the established evaluation outputs without changing model selection.
def metrics(model, frame):
    y = frame[target].to_numpy()
    p = predict(model, frame)
    labels = (p >= THRESHOLD).astype(int)
    return {"PR-AUC": average_precision_score(y, p), "Balanced Accuracy": balanced_accuracy_score(y, labels), "BCE": log_loss(y, p, labels=[0, 1]), "F1": f1_score(y, labels, zero_division=0), "AUROC": roc_auc_score(y, p)}
def forget_behaviour(model, frame):
    y = frame[target].to_numpy()
    p = np.clip(predict(model, frame), 1e-12, 1 - 1e-12)
    bce = -(y * np.log(p) + (1 - y) * np.log(1 - p))
    return {"Mean probability": float(p.mean()), "Positive prediction rate": float((p >= THRESHOLD).mean()), "Mean BCE": float(bce.mean())}


In [31]:

def apply_pending_request():
    global current_model, pending_request, request_number
    # Only confirmed requests are allowed to change model weights.
    complete_scope = pending_request["complete_scope"]
    training_forget = pending_request["training_forget"]
    request_text = pending_request["user_message"]
    new_ids = set(training_forget["assessment_id"])
    all_forgotten = forgotten_training_ids | new_ids
    all_scope = complete_scope_ids | set(complete_scope["assessment_id"])
    # Previously forgotten rows remain excluded from later retain updates.
    retain = assessments[assessments["original_split"].eq("train") & ~assessments["assessment_id"].isin(all_forgotten)].copy()
    retained_test = assessments[assessments["original_split"].eq("test") & ~assessments["assessment_id"].isin(all_scope)].copy()
    assert set(training_forget["assessment_id"]).isdisjoint(retain["assessment_id"])
    updated_model, runtime_seconds = run_gradient_difference(current_model, training_forget, retain)
    torch.save({"model_state_dict": updated_model.state_dict(), "objective": "-L_forget + L_retain", "epochs": 5}, MODEL_DIR / f"request_{request_number:03d}_epoch_5.pt")
    before = metrics(current_model, retained_test)
    after = metrics(updated_model, retained_test)
    results = pd.DataFrame({"Metric": list(before), "Before": list(before.values()), "After": list(after.values())})
    results["Change"] = results["After"] - results["Before"]
    before_forget = forget_behaviour(current_model, training_forget)
    after_forget = forget_behaviour(updated_model, training_forget)
    forget_results = pd.DataFrame({"Metric": list(before_forget), "Before": list(before_forget.values()), "After": list(after_forget.values())})
    forget_results["Change"] = forget_results["After"] - forget_results["Before"]
    summary = {"request": request_text, "complete_scope": len(complete_scope), "training_forget": len(training_forget), "runtime_seconds": runtime_seconds, "status": "completed"}
    results.to_csv(RESULT_DIR / f"request_{request_number:03d}_utility.csv", index=False)
    forget_results.to_csv(RESULT_DIR / f"request_{request_number:03d}_forget.csv", index=False)
    (RESULT_DIR / f"request_{request_number:03d}_summary.json").write_text(json.dumps(summary, indent=2))
    current_model = updated_model
    forgotten_training_ids.update(new_ids)
    complete_scope_ids.update(complete_scope["assessment_id"])
    request_history.append(summary)
    request_number += 1
    pending_request = None
    trusted_result = {"summary": summary, "retained_test_metrics": {row["Metric"]: {"before": row["Before"], "after": row["After"], "change": row["Change"]} for _, row in results.iterrows()}, "forget_set_behaviour": {row["Metric"]: {"before": row["Before"], "after": row["After"], "change": row["Change"]} for _, row in forget_results.iterrows()}}
    return summary, results, forget_results, trusted_result


In [32]:

def current_session_status():
    difference = float(np.max(np.abs(predict(current_model, verification_rows) - baseline_predictions)))
    return {"completed_deletion_requests": len(request_history), "unique_forgotten_training_rows": len(forgotten_training_ids), "model_state": "Original baseline" if difference == 0.0 else "Updated", "max_prediction_difference_vs_baseline": difference, "pending_request": pending_request is not None, "deletion_history": [item["request"] for item in request_history]}

def reset_session():
    # Reset reloads the frozen baseline rather than trying to reverse previous gradients.
    model = load_baseline_model()
    difference = float(np.max(np.abs(predict(model, verification_rows) - baseline_predictions)))
    assert difference == 0.0
    return model, difference


In [33]:
# Fail early if this stage no longer matches the frozen experiment contract.

def perform_verified_reset():
    global current_model, pending_request, request_number
    before = current_session_status()
    current_model, reset_difference = reset_session()
    forgotten_training_ids.clear()
    complete_scope_ids.clear()
    request_history.clear()
    pending_request = None
    request_number = 1
    after = current_session_status()
    assert reset_difference == 0.0 and after["unique_forgotten_training_rows"] == 0
    proof = pd.DataFrame({"State": ["Completed deletion requests", "Forgotten training rows", "Model state", "Max prediction difference vs baseline"], "Before reset": [before["completed_deletion_requests"], before["unique_forgotten_training_rows"], before["model_state"], before["max_prediction_difference_vs_baseline"]], "After reset": [after["completed_deletion_requests"], after["unique_forgotten_training_rows"], after["model_state"], after["max_prediction_difference_vs_baseline"]]})
    return proof, after


## 9. Start the conversation
Use the single `You:` prompt for conversation, dataset questions, deletion review, confirmation, cancellation, status and reset. Type `exit` when finished.


In [34]:
def handle_dataset_query_action(validated_action):
    # Python calculates trusted values before Qwen explains them.
    python_result = execute_dataset_query(validated_action, assessments)
    display(pd.DataFrame([python_result]))
    explanation = explain_and_show(
        "Answer the user's dataset question using the trusted result.",
        python_result,
    )
    return python_result, False, None, None, [explanation]


In [35]:

def show_pending_forget_request(pending_request, python_result):
    # Show trusted membership counts before the user is allowed to confirm.
    forget_set_size = int(len(pending_request["training_forget"]))
    print("Complete matching records:", python_result["complete_matching_records"])
    print(
        "Training records that influenced the model:",
        python_result["training_records_that_influenced_model"],
    )
    preview_columns = [
        "assessment_id", "recipient_id", "donor_id", "hospital_id",
        "assessment_date", "creatinine_mg_dl", target,
    ]
    display(pending_request["complete_scope"][preview_columns].head(10))
    explanation = explain_and_show(
        "Explain the trusted forget-set counts and ask whether to proceed with machine unlearning.",
        python_result,
    )
    return forget_set_size, explanation


In [36]:

def handle_forget_action(user_message, validated_action):
    global pending_request
    # Keep the pending request unchanged until the user confirms or cancels it.
    if pending_request is not None:
        python_result = {
            "status": "stopped",
            "reason": "Confirm or cancel the current pending request before creating another one.",
        }
        explanation = explain_and_show(
            "Explain why a second forget request cannot replace the pending request.",
            python_result,
        )
        return python_result, False, None, None, [explanation]
    pending_request, python_result = prepare_forget_request(
        user_message,
        validated_action,
        assessments,
    )
    if pending_request is None:
        explanation = explain_and_show(
            "Explain why Python did not create a pending forget set.",
            python_result,
        )
        return python_result, False, None, None, [explanation]
    forget_set_size, explanation = show_pending_forget_request(
        pending_request,
        python_result,
    )
    return python_result, False, forget_set_size, None, [explanation]


In [37]:

def handle_confirmation_action():
    # Only a confirmed pending request may call Gradient Difference.
    if pending_request is None:
        python_result = {
            "status": "stopped",
            "reason": "There is no pending forget request to confirm.",
        }
        explanation = explain_and_show(
            "Explain that there is no pending request.",
            python_result,
        )
        return python_result, False, None, None, [explanation]
    forget_set_size = int(len(pending_request["training_forget"]))
    summary, utility_results, forget_results, python_result = apply_pending_request()
    runtime_seconds = summary["runtime_seconds"]
    display(utility_results)
    display(forget_results)
    display(pd.DataFrame([summary]))
    explanation = explain_and_show(
        "Explain the completed Gradient Difference update using only these trusted results, then invite another conversational turn.",
        python_result,
    )
    return python_result, True, forget_set_size, runtime_seconds, [explanation]


In [38]:

def handle_cancel_action():
    global pending_request
    # Cancellation discards only the pending request and never changes model weights.
    cancelled_rows = 0 if pending_request is None else int(len(pending_request["training_forget"]))
    pending_request = None
    python_result = {
        "status": "cancelled",
        "cancelled_training_rows": cancelled_rows,
        "model_weights_changed": False,
    }
    explanation = explain_and_show(
        "Confirm that the pending request was cancelled without changing model weights.",
        python_result,
    )
    return python_result, False, None, None, [explanation]


In [39]:

def handle_status_action():
    # Session status is calculated from trusted Python state rather than conversation text.
    python_result = current_session_status()
    status_row = {
        key: value
        for key, value in python_result.items()
        if key != "deletion_history"
    }
    display(pd.DataFrame([status_row]))
    if python_result["deletion_history"]:
        display(pd.DataFrame({
            "Completed deletion request": python_result["deletion_history"],
        }))
    explanation = explain_and_show(
        "Describe the current trusted session state.",
        python_result,
    )
    return python_result, False, None, None, [explanation]


In [40]:

def handle_reset_action():
    # Reset proof is produced before Qwen describes the verified state change.
    state_before_reset = current_session_status()
    reset_proof, python_result = perform_verified_reset()
    model_changed = state_before_reset["model_state"] != "Original baseline"
    display(reset_proof)
    explanation = explain_and_show(
        "Explain the verified reset to the original frozen baseline.",
        python_result,
    )
    return python_result, model_changed, None, None, [explanation]


In [41]:

def dispatch_validated_action(user_message, validated_action):
    # Route only validated actions to their matching trusted Python operation.
    action = validated_action["action"]
    if action in {"chat", "help"}:
        return {"status": "conversation_only"}, False, None, None, []
    if action == "dataset_query":
        return handle_dataset_query_action(validated_action)
    if action == "forget":
        return handle_forget_action(user_message, validated_action)
    if action == "confirm":
        return handle_confirmation_action()
    if action == "cancel":
        return handle_cancel_action()
    if action == "session_status":
        return handle_status_action()
    if action == "reset":
        return handle_reset_action()
    return {"status": "conversation_closed"}, False, None, None, []


In [42]:

def finish_conversation_turn(user_message, raw_instruction, validated_action,
                             python_result, model_changed, forget_set_size,
                             runtime_seconds, assistant_messages):
    global turn_number
    # Keep a compact audit record separate from the visible conversation.
    record_turn(
        user_message,
        raw_instruction,
        validated_action,
        python_result,
        model_changed,
        forget_set_size,
        runtime_seconds,
    )
    conversation_history.append({"role": "user", "content": user_message})
    conversation_history.append({
        "role": "assistant",
        "content": "\n".join(assistant_messages),
    })
    turn_number += 1


In [43]:

def handle_qwen_failure(user_message, error):
    global turn_number
    # A failed language request must leave data and model state untouched.
    message = (
        "I could not interpret that turn because the Qwen request failed: "
        f"{type(error).__name__}. No data or model state was changed."
    )
    show_qwen(message)
    python_result = {
        "status": "qwen_error",
        "error_type": type(error).__name__,
    }
    record_turn(user_message, None, None, python_result, False, None)
    conversation_history.extend([
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": message},
    ])
    turn_number += 1


In [44]:

def display_interpreted_turn(raw_instruction):
    # Keep Qwen's natural reply primary and its structured action secondary.
    response_value = raw_instruction.get("response") if isinstance(raw_instruction, dict) else None
    natural_response = (
        response_value
        if isinstance(response_value, str)
        else "I could not produce a valid conversational response."
    )
    show_qwen(natural_response)
    print("\nStructured action retained for audit:")
    if isinstance(raw_instruction, dict):
        display({
            "action": raw_instruction.get("action"),
            "parameters": raw_instruction.get("parameters"),
        })
    return [natural_response]


In [45]:

def stopped_validation_result(validation_errors):
    # Invalid actions stop before any trusted data or model operation runs.
    python_result = {
        "status": "stopped",
        "validation_errors": validation_errors,
    }
    print("Python validation stopped this turn:", *validation_errors)
    return python_result, False, None, None, []


In [46]:

def run_conversation_turn(user_message):
    # Qwen handles language first; Python then validates and dispatches the action.
    try:
        raw_instruction = interpret_turn(user_message)
    except Exception as error:
        handle_qwen_failure(user_message, error)
        return False
    assistant_messages = display_interpreted_turn(raw_instruction)
    validated_action = normalise_action(raw_instruction)
    validated_action = normalise_entity_value(validated_action, assessments)
    validation_errors = validate_action(validated_action, assessments)
    if validation_errors:
        action_result = stopped_validation_result(validation_errors)
    else:
        action_result = dispatch_validated_action(user_message, validated_action)
    python_result, model_changed, forget_set_size, runtime_seconds, extra_messages = action_result
    assistant_messages.extend(extra_messages)
    finish_conversation_turn(
        user_message,
        raw_instruction,
        validated_action,
        python_result,
        model_changed,
        forget_set_size,
        runtime_seconds,
        assistant_messages,
    )
    return (
        isinstance(validated_action, dict)
        and validated_action.get("action") == "exit"
        and not validation_errors
    )


In [ ]:

# Keep one conversational prompt for chat, queries, confirmation and reset.
print("Qwen: Hi! I can help you explore the kidney-transplant experiment, prepare a forget set, show session status or reset to the original baseline.")
print("Type 'exit' when you want to finish.")
while True:
    user_message = input("\nYou: ").strip()
    if not user_message:
        continue
    if run_conversation_turn(user_message):
        break


Qwen: Hi! I can help you explore the kidney-transplant experiment, prepare a forget set, show session status or reset to the original baseline.
Type 'exit' when you want to finish.

Qwen:
Hello. I am the conversational interface for the kidney-transplant machine-unlearning notebook. How can I assist you today?

Structured action retained for audit:


{'action': 'chat', 'parameters': None}

## 10. Findings and limitations
Qwen provides the natural-language interface and preserves multi-turn context. Python validates actions, calculates trusted statistics, constructs forget sets and verifies session changes. Confirmation through the same conversation gates the unchanged Gradient Difference update.

The data are synthetic, supported requests are narrow, Qwen may misinterpret language, Gradient Difference is approximate, feature filters are exploratory, and sequential updates may accumulate utility loss.


# Results — Current Deletion Request

## 1. Request Summary

| Item | Result |
| --- | ---: |
| Natural-language request | Remove all data for donor V32P-DL000539 |
| Interpreted deletion | Exact donor_id match: V32P-DL000539 |
| Complete matching rows | 6 |
| Training rows forgotten | 6 |
| Unlearning method | Gradient Difference |

## 2. Utility

| Metric | Before Request | After Unlearning | Change |
| --- | ---: | ---: | ---: |
| PR-AUC ↑ | 0.1689 | 0.1689 | 0.0000 |
| Balanced Accuracy ↑ | 0.6304 | 0.6304 | 0.0000 |
| BCE ↓ | 0.5751 | 0.5751 | 0.0000 |
| F1 ↑ | 0.2446 | 0.2446 | 0.0000 |
| AUROC ↑ | 0.7321 | 0.7321 | -0.0000 |
| Composite Utility ↑ | 0.3392 | 0.3392 | -0.0000 |

Composite Utility is a supporting summary metric; individual utility metrics remain visible.

## 3. Forget-Set Behaviour

| Diagnostic | Before Request | After Unlearning | Change |
| --- | ---: | ---: | ---: |
| Mean Truth Ratio | Unavailable | Unavailable | Unavailable |
| Mean BCE | 0.4381 | 0.4396 | 0.0016 |
| Mean Predicted Probability | 0.3370 | 0.3379 | 0.0009 |

**Full-Retraining Reference: Unavailable**

No matching full-retraining reference exists for this dynamic forget set. The before/after forgetting values above are descriptive diagnostics only. A Truth Ratio was not saved for this request and is therefore reported as unavailable.

## 4. Efficiency

| Measure | Result |
| --- | ---: |
| Gradient Difference Runtime | 0.027 s |
| Historical Mean Full Retraining Runtime* | 3.558 s |
| Indicative Speed-Up* | 134.21× |

*Historical benchmark only. This is not an exact full retraining run for the dynamic request unless a matching reference exists.*

## Key Findings

- The latest completed request selected **6** donor-linked rows, all **6** of which belonged to training.
- The recorded Gradient Difference update took **0.027 seconds**.
- Utility changes were small for this request, but no exact Full Retraining reference or saved Truth Ratio is available.
- The before/after diagnostics alone do not demonstrate complete erasure.
